In [13]:
 
import numpy as np 
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import math
import pandas as pd 
import matplotlib.pyplot as plt

# import seaborn as sns
import sklearn.metrics as metrics
%matplotlib inline
import os
from pandas_ml import ConfusionMatrix

In [14]:
df_train = pd.read_csv("./data/df_train.csv",index_col=0)
# df_train2 = df_train.drop(df_train[df_train['attack_type'] == 1].index)
df_train_1 = df_train[df_train['attack_type'].isin([1])]
df_train_0 = df_train[df_train['attack_type'].isin([0])]

df_ma, df_mi = train_test_split(df_train_1, test_size=0.1, random_state=42)
df_train2 = df_train_0.append(df_mi)

X_train, X_test, y_train, y_test = train_test_split(df_train2.drop(['attack_type'],axis=1), df_train2['attack_type'], test_size=0.2, random_state=42)

In [15]:
data_x2 = df_train

In [16]:
y_train.max()

1

In [17]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(df_train2, df_train2['attack_type'], test_size=0.2, random_state=42)
X_train.shape

(67360, 40)

In [18]:
y_train.shape

(67360,)

In [19]:
Y_train = y_train
Y_test = y_test
dict = {}
for i in Y_train:
    dict.update({i:dict.get(i,0)+1})
dict

{0: 61648, 1: 5712}

In [20]:
Y_train.shape

(67360,)

In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Normalizer

In [22]:
X = X_train
T = X_test
Y = y_train
C = y_test

scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

scaler = Normalizer().fit(T)
testT = scaler.transform(T)

traindata = np.array(trainX)
trainlabel = np.array(Y)

testdata = np.array(testT)
testlabel = np.array(C)
testlabel = testlabel.flatten()

In [23]:
testlabel.shape

(16841,)

In [24]:

KNN = KNeighborsClassifier()
KNN.fit(traindata, trainlabel)

DT = DecisionTreeClassifier()
DT.fit(traindata, trainlabel)

RF = RandomForestClassifier(n_estimators=100)
RF.fit(traindata, trainlabel)

RandomForestClassifier(bootstrap=True, class_weight=None, criterion='gini',
                       max_depth=None, max_features='auto', max_leaf_nodes=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=100,
                       n_jobs=None, oob_score=False, random_state=None,
                       verbose=0, warm_start=False)

In [25]:
np.seterr(invalid='ignore')

{'divide': 'warn', 'invalid': 'warn', 'over': 'warn', 'under': 'ignore'}

In [26]:

nums = 4  #分块数量

In [27]:
#-*- coding: utf8
from __future__ import division, print_function

import numpy as np

def _compute_centroids(X, assign, num_clusters):
    C = np.zeros(shape=(num_clusters, X.shape[1]), dtype='d')
    for k in range(num_clusters):

        if not (assign == k).any():
            continue

        K = X[assign == k]
        if K.ndim == 1:
            K = K[np.newaxis]
        C[k] = X[assign == k].mean(axis=0) 
    return C

def _surprisal_mat(X):
    #Some elements have zero prob, ingore and treat warnings
    with np.errstate(divide='ignore', invalid='ignore'):
        L = np.log2(X)
        L[np.isnan(L)] = 0
        L[np.isinf(L)] = 0
    return L

def _dist_all(X, C):
    S_x = _surprisal_mat(X)
    S_c = _surprisal_mat(C)
    
    D = (X * (S_x - S_c[:, np.newaxis,: ])).sum(axis=2).T
    return D

def _base_kmeans(X, C, n_iters=-1):
    
    num_clusters = C.shape[0]
    n = X.shape[0]

    C_final = C

    #KMeans algorithm
    cent_dists = None
    assign = None
    prev_assign = None
    best_shift = None

    iters = n_iters
    converged = False

    while iters != 0 and not converged:
        #assign elements to new clusters    
        D = _dist_all(X, C)
        assign = D.argmin(axis=1)
        
        #check if converged, if not compute new centroids
        if prev_assign is not None and not (prev_assign - assign).any():
            converged = True
        else: 
            C_final = _compute_centroids(X, assign, num_clusters)

        prev_assign = assign
        iters -= 1
    
    return C_final, assign

def cost(X, C, assign):
    cost = 0
    for k in set(assign):
        idx = assign == k
        cost += _dist_all(X[idx], C[k][np.newaxis]).sum()
    return cost

def klkmeans(X, num_clusters, n_iters=-1, n_runs=10):

    min_cost = float('+inf')
    best_C = None
    best_assign = None

    for _ in range(n_runs):
        assign = np.random.randint(0, num_clusters, X.shape[0])
        C = _compute_centroids(X, assign, num_clusters)

        C, assign = _base_kmeans(X, C, n_iters)
        clust_cost = cost(X, C, assign)

        if clust_cost < min_cost:
            best_C = C
            best_assign = assign

    return best_C, best_assign

if __name__ == '__main__':
    np.seterr(all='raise')
#     X = np.zeros((200, 1000))
#     X[0:100] = 1
#     X[100:200, 500:] = 1
#     X += 1e-20
    
#     X = (X.T / X.sum(axis=1)).T
#     C, assign = klkmeans(X, 2)
    
#     assert ((C.sum(axis=1) - 1) < 1e-10).all()
#     assert (assign[0:100] != assign[100:]).all()
    
    import os
#     dir_ = os.path.dirname('D:\mycode\CycleGAN_MetaLearning\src1_NSL-KDD')
#     fpath = os.path.dirname( './testdata.dat')
#     fpath = './testdata.dat'
#     X = np.genfromtxt(fpath)


    X = data_x2.values
    
    
    C, assign = klkmeans(X, nums) 
    assert len(set(assign)) == nums

  
#     for nums in range(10): 
#         nums += 1        
#         C, assign = klkmeans(X, nums)
#   #      C2, assign2 = klkmeans(X2, nums)
#         print(nums)
#         if len(set(assign)) == nums :
#             print("----"+str(nums))

In [28]:
assign_ = assign.reshape(len(assign),1)
X_ = np.concatenate((X,assign_),axis = 1)

# assign2_ = assign2.reshape(len(assign2),1)
# X2_ = np.concatenate((X2,assign2_),axis = 1)

In [29]:
assign

array([2, 3, 0, ..., 2, 1, 3], dtype=int64)

In [30]:
print(X_.shape)
# print(X2_.shape)

(148517, 42)


In [31]:
CC={}
CT={}
for i in range(nums):
    
    X_df = X_[X_[...,-1]==i,:-1]
    y_df= X_[X_[...,-1]==i,-1]

    CC[i], CT[i], y_tr, y_te = train_test_split(X_df, y_df, test_size=0.2, random_state=42)
    print(CC[i].shape,end = " ")
    print(CT[i].shape)

(34949, 41) (8738, 41)
(13759, 41) (3440, 41)
(45355, 41) (11339, 41)
(24749, 41) (6188, 41)


In [32]:
# np.set_printoptions(suppress=True)
# list = np.zeros(4)
# for i in assign:
#     list[i] += 1
# list

In [33]:
knn = []
dt = []
rf = []
for i in range(nums):
    knn_ = KNeighborsClassifier()
    knn_.fit(CC[i][...,:-1], CC[i][...,-1].flatten())
    knn.append(knn_)
    
    dt_ = DecisionTreeClassifier()
    dt_.fit(CC[i][...,:-1], CC[i][...,-1].flatten())
    dt.append(dt_)

    rf_ = RandomForestClassifier(n_estimators=100)
    rf_.fit(CC[i][...,:-1], CC[i][...,-1].flatten())
    rf.append(rf_)
    
#     expected = CT[i][...,-1].flatten()
#     predicted = knn_.predict(CT[i][...,:-1])
#     # summarize the fit of the model
#     cm = ConfusionMatrix(expected, predicted)
#     print(cm)
#     try:
# #         np.errstate(divide="ignore")
#         cm.stats()
#     except:
#         continue
# #     print(cm.to_dataframe().loc['actual' = 'False', 'predicted' = 'False'])
#     print(cm(super.get(actual = 'False', predicted = 'False'))

In [34]:
# k= 3
# assign = np.random.randint(0, 4, 10)
# idx = assign == k
# print(idx)
# print(assign)


In [35]:
from pymoo.core.problem import Problem
class MyProblem(Problem):
    def __init__(self):
        self.i = i
        self.P1 = P1
        self.P2 = P2
        self.P3 = P3
        self.TP1 = TP1
        self.TP2 = TP2
        self.TP3 = TP3
        self.TN1 = TN1
        self.TN2 = TN2
        self.TN3 = TN3
        super().__init__(n_var=3,   # 变量数
                         n_obj=1,   # 目标数
                         n_constr=2,    # 约束数
                         xl=np.array([0,0,0]),     # 变量下界
                         xu=np.array([1,1,1]),   # 变量上界
                         )

    def _evaluate(self, x, out, *args, **kwargs):

        # 定义目标函数
#         f = dict([(key,[]) for key in range(4)])
#         f = dict.fromkeys(range(0,4),[])
#         print(i+1)
        
        f = 1 - (TP1 + TN1) * x[:,0] / P1  - (TP2 + TN2) * x[:,1] / P2 - (TP3 + TN3) * x[:,2] / P3 
    # 定义约束条件
        g1 = x[:,0] + x[:,1] + x[:,2] - 1
        g2 = - x[:,0] - x[:,1] - x[:,2] + 0.9
        # todo
        out["F"] = np.column_stack([f])
        out["G"] = np.column_stack([g1,g2])
        
#         print(cm1)
#         print(cm2)
#         print(cm3)
#         print()



In [36]:
# pd.crosstab(cm1._y_true, cm1._y_pred).reindex()[1][1]
# pd.crosstab(cm1._y_true, cm1._y_pred).iloc[1][1]

In [37]:

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.factory import get_sampling, get_crossover, get_mutation
from pymoo.optimize import minimize
# from example import MyProblem

# 定义遗传算法
algorithm = NSGA2(
    pop_size=40,
    n_offsprings=10,
    sampling=get_sampling("real_random"),
    crossover=get_crossover("real_sbx", prob=0.9, eta=15),
    mutation=get_mutation("real_pm", eta=20),
    eliminate_duplicates=True
)



Compiled modules for significant speedup can not be used!
https://pymoo.org/installation.html#installation

To disable this warning:
from pymoo.config import Config
Config.show_compile_hint = False



In [38]:
ans = []
for i in range(nums):
    print(i)
    
    predicted1 = knn[i].predict(CC[i][...,:-1])
    predicted2 = dt[i].predict(CC[i][...,:-1])
    predicted3 = rf[i].predict(CC[i][...,:-1])
    cm1 = ConfusionMatrix(CC[i][...,-1], predicted1)
    cm2 = ConfusionMatrix(CC[i][...,-1], predicted2)
    cm3 = ConfusionMatrix(CC[i][...,-1], predicted3)

    a = pd.crosstab(cm1._y_true, cm1._y_pred)
    TP1 = a.reindex()[0][0]
    TN1 = a.reindex()[1][1]
    P1 = cm1.population

    b = pd.crosstab(cm2._y_true, cm2._y_pred)
    TP2 = b.reindex()[0][0]
    TN2 = b.reindex()[1][1]
    P2 = cm2.population

    c = pd.crosstab(cm3._y_true, cm3._y_pred)
    TP3 = c.reindex()[0][0]
    TN3 = c.reindex()[1][1]
    P3= cm3.population
    
    res = minimize(MyProblem(),
                   algorithm,
                   ('n_gen', 20),
                   seed=1,
                   verbose=True
                   )
    ans.append(res)
    


0
n_gen |  n_eval |   cv (min)   |   cv (avg)   |  n_nds  |     eps      |  indicator  
    1 |      40 |  0.00000E+00 |  0.550933661 |       1 |            - |            -
    2 |      50 |  0.00000E+00 |  0.381130969 |       1 |  0.035986937 |        ideal
    3 |      60 |  0.00000E+00 |  0.319989816 |       1 |  0.00000E+00 |            f
    4 |      70 |  0.00000E+00 |  0.234952351 |       1 |  0.00000E+00 |            f
    5 |      80 |  0.00000E+00 |  0.167437617 |       1 |  0.00000E+00 |            f
    6 |      90 |  0.00000E+00 |  0.114886963 |       1 |  0.00000E+00 |            f
    7 |     100 |  0.00000E+00 |  0.100957779 |       1 |  0.00000E+00 |            f
    8 |     110 |  0.00000E+00 |  0.081946271 |       1 |  0.00000E+00 |            f
    9 |     120 |  0.00000E+00 |  0.057463230 |       1 |  0.034855295 |        ideal
   10 |     130 |  0.00000E+00 |  0.044006964 |       1 |  0.00000E+00 |            f
   11 |     140 |  0.00000E+00 |  0.026718023 |     

In [39]:
ans[0].X

array([0.41417927, 0.04995346, 0.53583676])

In [40]:
for j in range(nums):
    
    print(ans[j].X)

[0.41417927 0.04995346 0.53583676]
[0.41417927 0.04995346 0.53583676]
[0.41417927 0.04995346 0.53583676]
[0.41417927 0.04995346 0.53583676]


In [41]:
ans[1].X[1]

0.04995345894608716

In [44]:
expected = testlabel
np.savetxt("Exp.txt", expected) 

predicted1 = KNN.predict(testdata)
predicted2 =  DT.predict(testdata)
predicted3 = RF.predict(testdata)

cm1 = ConfusionMatrix(expected, predicted1)
np.savetxt("Pre1.txt", predicted1) 

cm2 = ConfusionMatrix(expected, predicted2)
np.savetxt("Pre2.txt", predicted2) 

cm3 = ConfusionMatrix(expected, predicted3)
np.savetxt("Pre3.txt", predicted3) 

cm1.stats()

OrderedDict([('population', 16841),
             ('P', 1435),
             ('N', 15406),
             ('PositiveTest', 1385),
             ('NegativeTest', 15456),
             ('TP', 1359),
             ('TN', 15380),
             ('FP', 26),
             ('FN', 76),
             ('TPR', 0.9470383275261324),
             ('TNR', 0.9983123458392834),
             ('PPV', 0.9812274368231046),
             ('NPV', 0.9950828157349897),
             ('FPR', 0.0016876541607166039),
             ('FDR', 0.018772563176895306),
             ('FNR', 0.0529616724738676),
             ('ACC', 0.99394335253251),
             ('F1_score', 0.9638297872340426),
             ('MCC', 0.9607057586323471),
             ('informedness', 0.9453506733654158),
             ('markedness', 0.9763102525580942),
             ('prevalence', 0.08520871682204145),
             ('LRP', 561.1566336102921),
             ('LRN', 0.053051204559974266),
             ('DOR', 10577.641700404858),
             ('FOR', 0.004

In [45]:
cm2.stats()

OrderedDict([('population', 16841),
             ('P', 1435),
             ('N', 15406),
             ('PositiveTest', 1439),
             ('NegativeTest', 15402),
             ('TP', 1386),
             ('TN', 15353),
             ('FP', 53),
             ('FN', 49),
             ('TPR', 0.9658536585365853),
             ('TNR', 0.9965597819031546),
             ('PPV', 0.9631688672689368),
             ('NPV', 0.9968185949876639),
             ('FPR', 0.003440218096845385),
             ('FDR', 0.03683113273106324),
             ('FNR', 0.03414634146341464),
             ('ACC', 0.99394335253251),
             ('F1_score', 0.964509394572025),
             ('MCC', 0.9611996859806967),
             ('informedness', 0.96241344043974),
             ('markedness', 0.9599874622566007),
             ('prevalence', 0.08520871682204145),
             ('LRP', 280.75361251725724),
             ('LRN', 0.03426421784572174),
             ('DOR', 8193.784366576818),
             ('FOR', 0.00318140

In [46]:
cm3.stats()

OrderedDict([('population', 16841),
             ('P', 1435),
             ('N', 15406),
             ('PositiveTest', 1390),
             ('NegativeTest', 15451),
             ('TP', 1381),
             ('TN', 15397),
             ('FP', 9),
             ('FN', 54),
             ('TPR', 0.9623693379790941),
             ('TNR', 0.9994158120212904),
             ('PPV', 0.9935251798561151),
             ('NPV', 0.9965050805773089),
             ('FPR', 0.0005841879787095936),
             ('FDR', 0.006474820143884892),
             ('FNR', 0.037630662020905925),
             ('ACC', 0.9962591295053738),
             ('F1_score', 0.9776991150442478),
             ('MCC', 0.9758055147086844),
             ('informedness', 0.9617851500003844),
             ('markedness', 0.9900302604334241),
             ('prevalence', 0.08520871682204145),
             ('LRP', 1647.3624467673249),
             ('LRN', 0.03765265825122275),
             ('DOR', 43751.55761316873),
             ('FOR', 0.0

In [47]:
expectedEnd = np.empty(shape = [0,1]).flatten()
predictedEnd = np.empty(shape = [0,1]).flatten()
for i in range(nums):
    expected2 = CT[i][...,-1]
    testdata2 = CT[i][...,:-1]
    predicted2 = np.around(ans[i].X[0] * knn[i].predict(testdata2) +  ans[i].X[1] * dt[i].predict(testdata2) + ans[i].X[2] * rf[i].predict(testdata2))
#     predicted2 = np.around(0.09 * knn[i].predict(testdata2) +  0.3 * dt[i].predict(testdata2) + 0.2 * rf[i].predict(testdata2))
    expectedEnd = np.concatenate((expectedEnd, expected2), axis = 0)
    predictedEnd = np.concatenate((predictedEnd,predicted2), axis = 0)
    
    print(expectedEnd.shape)
    print(predictedEnd.shape)

cm = ConfusionMatrix(expectedEnd, predictedEnd)
expected = np.array(expectedEnd)
predicted = np.array(predictedEnd)

np.savetxt("AdassPre.txt", predicted) 
np.savetxt("AdassExp.txt", expected) 

cm.stats()


(8738,)
(8738,)
(12178,)
(12178,)
(23517,)
(23517,)
(29705,)
(29705,)


OrderedDict([('population', 29705),
             ('P', 14282),
             ('N', 15423),
             ('PositiveTest', 14268),
             ('NegativeTest', 15437),
             ('TP', 14216),
             ('TN', 15371),
             ('FP', 52),
             ('FN', 66),
             ('TPR', 0.9953787984876068),
             ('TNR', 0.9966284121117811),
             ('PPV', 0.9963554807961873),
             ('NPV', 0.9957245578804171),
             ('FPR', 0.003371587888218894),
             ('FDR', 0.0036445192038127277),
             ('FNR', 0.004621201512393223),
             ('ACC', 0.99602760478034),
             ('F1_score', 0.9958669001751314),
             ('MCC', 0.9920436239696878),
             ('informedness', 0.9920072105993878),
             ('markedness', 0.9920800386766044),
             ('prevalence', 0.480794479043932),
             ('LRP', 295.22552325143),
             ('LRN', 0.0046368350091497415),
             ('DOR', 63669.62004662004),
             ('FOR', 0.00